In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
data = pd.read_csv('/content/drive/My Drive/UTS_Kasus08/kasus08_infert.csv')
print(data)


    education  age  parity  induced  case  spontaneous  stratum  \
0      0-5yrs   26       6        1     1            2        1   
1      0-5yrs   42       1        1     1            0        2   
2      0-5yrs   39       6        2     1            0        3   
3      0-5yrs   34       4        2     1            0        4   
4     6-11yrs   35       3        1     1            1        5   
..        ...  ...     ...      ...   ...          ...      ...   
243   12+ yrs   31       1        0     0            1       79   
244   12+ yrs   34       1        0     0            0       80   
245   12+ yrs   35       2        2     0            0       81   
246   12+ yrs   29       1        0     0            1       82   
247   12+ yrs   23       1        0     0            1       83   

     pooled.stratum  
0                 3  
1                 1  
2                 4  
3                 2  
4                32  
..              ...  
243              45  
244              47

In [ ]:
import pandas as pd

data = pd.read_csv('/content/drive/My Drive/UTS_Kasus08/kasus08_infert.csv')
print("Ukuran data:", data.shape)
print("\nCek Missing Value:\n", data.isna().sum())
print("\nJumlah Duplikat Penuh:", data.duplicated().sum())
print("\nDistribusi Target (case):\n", data["case"].value_counts())
data.head()

Ukuran data: (248, 8)

Cek Missing Value:
 education         0
age               0
parity            0
induced           0
case              0
spontaneous       0
stratum           0
pooled.stratum    0
dtype: int64

Jumlah Duplikat Penuh: 31

Distribusi Target (case):
 case
0    165
1     83
Name: count, dtype: int64


,education,age,parity,induced,case,spontaneous,stratum,pooled.stratum
0,0-5yrs,26,6,1,1,2,1,3
1,0-5yrs,42,1,1,1,0,2,1
2,0-5yrs,39,6,2,1,0,3,4
3,0-5yrs,34,4,2,1,0,4,2
4,6-11yrs,35,3,1,1,1,5,32


In [ ]:
%%writefile fitur.py
"""Modul feature engineering untuk Kasus 8: Studi Infertilitas (NIM 23552011090)."""
import numpy as np
import pandas as pd

KOLOM_DIBUANG = ["stratum", "pooled.stratum"]
FITUR_MENTAH = ["education", "age", "parity", "induced", "spontaneous"]
TARGET_KOLOM = "case"

URUTAN_PENDIDIKAN = ["0-5yrs", "6-11yrs", "12+ yrs"]

FITUR_NUMERIK = [
    "age",
    "parity",
    "induced",
    "spontaneous",
    "total_kehamilan",
    "total_kehilangan",
    "rasio_kehilangan",
    "rasio_spontan",
]
FITUR_ORDINAL = ["education"]


def bersihkan_teks(data: pd.DataFrame) -> pd.DataFrame:
    """Membersihkan spasi berlebih pada kolom kategorik."""
    out = data.copy()
    if "education" in out.columns:
        out["education"] = out["education"].astype(str).str.strip()
    return out


def buat_fitur_turunan(data: pd.DataFrame) -> pd.DataFrame:
    """Menambahkan fitur turunan dari riwayat kehamilan (stateless, bebas data leakage)."""
    out = bersihkan_teks(data)
    out["total_kehamilan"] = out["parity"] + out["induced"] + out["spontaneous"]
    out["total_kehilangan"] = out["induced"] + out["spontaneous"]

    denom = out["total_kehamilan"].replace(0, np.nan)
    out["rasio_kehilangan"] = (out["total_kehilangan"] / denom).fillna(0.0)
    out["rasio_spontan"] = (out["spontaneous"] / denom).fillna(0.0)
    return out

Writing fitur.py


In [ ]:
%%writefile latih.py
"""Skrip pelatihan, evaluasi, dan penyimpanan model Kasus 8: Studi Infertilitas."""
from pathlib import Path
import joblib
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder, StandardScaler

from fitur import (
    FITUR_MENTAH,
    FITUR_NUMERIK,
    FITUR_ORDINAL,
    KOLOM_DIBUANG,
    TARGET_KOLOM,
    URUTAN_PENDIDIKAN,
    buat_fitur_turunan,
)

RANDOM_STATE = 42
BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = "/content/drive/My Drive/UTS_Kasus08/kasus08_infert.csv"
MODEL_DIR = BASE_DIR / "models"
MODEL_PATH = MODEL_DIR / "model_infertilitas_kasus08.joblib"


def audit_data(df: pd.DataFrame) -> dict:
    """Melakukan audit kualitas data awal (duplikat, missing value, nilai tidak wajar)."""
    return {
        "jumlah_baris": int(df.shape[0]),
        "jumlah_kolom": int(df.shape[1]),
        "duplikat_penuh": int(df.duplicated().sum()),
        "missing_total": int(df.isna().sum().sum()),
        "usia_tidak_wajar": int(((df["age"] < 15) | (df["age"] > 60)).sum()),
        "negatif_kehamilan": int(
            ((df["parity"] < 0) | (df["induced" ] < 0) | (df["spontaneous"] < 0)).sum()
        ),
    }


def bangun_pipeline(classifier) -> Pipeline:
    """Membangun pipeline lengkap dari input mentah hingga prediksi klasifikasi."""
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                FITUR_NUMERIK,
            ),
            (
                "ord",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                categories=[URUTAN_PENDIDIKAN],
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                            ),
                        ),
                    ]
                ),
                FITUR_ORDINAL,
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("feature_engineering", FunctionTransformer(buat_fitur_turunan)),
            ("preprocessing", preprocessor),
            ("classifier", classifier),
        ]
    )


def main():
    df = pd.read_csv(DATA_PATH)
    hasil_audit = audit_data(df)
    print("=== AUDIT DATA ===")
    for k, v in hasil_audit.items():
        print(f"- {k}: {v}")

    # Buang kolom desain studi (stratum, pooled.stratum) agar tidak terjadi data leakage
    df_model = df.drop(columns=[c for c in KOLOM_DIBUANG if c in df.columns])

    X = df_model[FITUR_MENTAH].copy()
    y = df_model[TARGET_KOLOM].astype(int)

    # Pembagian train & test SEBELUM imputasi/scaling/encoding (anti-leakage)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    print(f"\nData Train: {X_train.shape[0]} baris | Data Test: {X_test.shape[0]} baris")

    # Bandingkan model menggunakan Stratified 5-Fold CV pada data TRAIN saja
    kandidat = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200, max_depth=5, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE
        ),
        "KNN (k=7)": KNeighborsClassifier(n_neighbors=7, weights="distance"),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

    print("\n=== PERBANDINGAN MODEL (5-FOLD CV PADA DATA TRAIN) ===")
    ringkasan_cv = []
    for nama, clf in kandidat.items():
        pipe = bangun_pipeline(clf)
        scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
        ringkasan_cv.append(
            {
                "model": nama,
                "cv_accuracy": round(float(scores["test_accuracy"].mean()), 4),
                "cv_precision": round(float(scores["test_precision"].mean()), 4),
                "cv_recall": round(float(scores["test_recall"].mean()), 4),
                "cv_f1": round(float(scores["test_f1"].mean()), 4),
                "cv_roc_auc": round(float(scores["test_roc_auc"].mean()), 4),
            }
        )

    df_cv = pd.DataFrame(ringkasan_cv).sort_values(by="cv_roc_auc", ascending=False)
    print(df_cv.to_string(index=False))

    nama_terbaik = df_cv.iloc[0]["model"]
    print(f"\nModel terpilih berdasarkan CV: {nama_terbaik}")

    # Latih model terbaik pada seluruh data train & evaluasi SATU KALI pada data test
    model_final = bangun_pipeline(kandidat[nama_terbaik])
    model_final.fit(X_train, y_train)

    y_pred = model_final.predict(X_test)
    y_proba = model_final.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    metrik_test = {
        "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
        "balanced_accuracy": round(float(balanced_accuracy_score(y_test, y_pred)), 4),
        "precision": round(float(precision_score(y_test, y_pred)), 4),
        "recall": round(float(recall_score(y_test, y_pred)), 4),
        "f1_score": round(float(f1_score(y_test, y_pred)), 4),
        "roc_auc": round(float(roc_auc_score(y_test, y_proba)), 4),
        "pr_auc": round(float(average_precision_score(y_test, y_proba)), 4),
        "confusion_matrix": {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)},
    }

    print("\n=== EVALUASI AKHIR PADA DATA TEST (1 KALI) ===")
    for k, v in metrik_test.items():
        print(f"- {k}: {v}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["Kontrol (0)", "Kasus (1)"]))

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    bundle = {
        "pipeline": model_final,
        "nama_model": nama_terbaik,
        "fitur_mentah": FITUR_MENTAH,
        "opsi_education": URUTAN_PENDIDIKAN,
        "threshold": 0.5,
        "hasil_audit": hasil_audit,
        "perbandingan_cv": ringkasan_cv,
        "metrik_test": metrik_test,
        "versi_sklearn": sklearn.__version__,
    }
    joblib.dump(bundle, MODEL_PATH, compress=3)
    print(f"Model berhasil disimpan ke: {MODEL_PATH}")


if __name__ == "__main__":
    main()

Writing latih.py


In [ ]:
%run latih.py

=== AUDIT DATA ===
- jumlah_baris: 248
- jumlah_kolom: 8
- duplikat_penuh: 31
- missing_total: 0
- usia_tidak_wajar: 0
- negatif_kehamilan: 0

Data Train: 198 baris | Data Test: 50 baris

=== PERBANDINGAN MODEL (5-FOLD CV PADA DATA TRAIN) ===
              model  cv_accuracy  cv_precision  cv_recall  cv_f1  cv_roc_auc
Logistic Regression       0.6864        0.5297     0.6495 0.5795      0.7657
      Random Forest       0.7019        0.5500     0.5429 0.5412      0.7106
  Gradient Boosting       0.7067        0.5952     0.4209 0.4839      0.6689
          KNN (k=7)       0.7021        0.5893     0.4077 0.4780      0.6433

Model terpilih berdasarkan CV: Logistic Regression

=== EVALUASI AKHIR PADA DATA TEST (1 KALI) ===
- accuracy: 0.66
- balanced_accuracy: 0.6569
- precision: 0.5
- recall: 0.6471
- f1_score: 0.5641
- roc_auc: 0.7629
- pr_auc: 0.6731
- confusion_matrix: {'TN': 22, 'FP': 11, 'FN': 6, 'TP': 11}

Classification Report:
              precision    recall  f1-score   support



In [ ]:
%%writefile app.py
"""Aplikasi Web Streamlit untuk Prediksi Risiko Infertilitas (Kasus 8)."""
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

import fitur  # Wajib di-import agar FunctionTransformer di pipeline terbaca saat load

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "models" / "model_infertilitas_kasus08.joblib"


@st.cache_resource
def load_bundle():
    """Memuat bundle pipeline dan metadata model secara efisien."""
    if not MODEL_PATH.exists():
        return None
    return joblib.load(MODEL_PATH)


st.set_page_config(
    page_title="Prediksi Risiko Infertilitas - Kasus 8",
    layout="wide",
)

st.title("Sistem Klasifikasi Risiko Infertilitas (Kasus 8)")
st.caption("UTS Intelegensi Buatan · NIM 23552011090 · Dataset: kasus08_infert.csv")

bundle = load_bundle()

if bundle is None:
    st.error(
        "File model tidak ditemukan di `models/model_infertilitas_kasus08.joblib`. "
        "Jalankan `python latih.py` terlebih dahulu."
    )
    st.stop()

tab_prediksi, tab_evaluasi = st.tabs(["Prediksi Pasien Baru", "Evaluasi & Metadata Model"])

with tab_prediksi:
    st.subheader("Masukkan Data Riwayat Klinis & Kehamilan")
    col1, col2 = st.columns(2)

    with col1:
        education = st.selectbox(
            "Tingkat Pendidikan",
            options=bundle["opsi_education"],
            index=1,
            help="Kategori lama menempuh pendidikan formal.",
        )
        age = st.number_input("Usia (tahun)", min_value=15, max_value=60, value=31, step=1)
        parity = st.number_input(
            "Paritas (jumlah kelahiran hidup sebelumnya)",
            min_value=0,
            max_value=15,
            value=2,
            step=1,
        )

    with col2:
        induced = st.number_input(
            "Jumlah Aborsi yang Diinduksi (induced)",
            min_value=0,
            max_value=10,
            value=0,
            step=1,
        )
        spontaneous = st.number_input(
            "Jumlah Keguguran Spontan (spontaneous)",
            min_value=0,
            max_value=10,
            value=0,
            step=1,
        )

    total_kehamilan = parity + induced + spontaneous
    total_kehilangan = induced + spontaneous
    rasio_spontan = (spontaneous / total_kehamilan) if total_kehamilan > 0 else 0.0

    st.info(
        f"**Ringkasan Otomatis (Feature Engineering):** Total Kehamilan = **{total_kehamilan}** | "
        f"Total Kehilangan = **{total_kehilangan}** | Rasio Keguguran Spontan = **{rasio_spontan:.1%}**"
    )

    if st.button("Prediksi Risiko Infertilitas", type="primary"):
        input_df = pd.DataFrame(
            [
                {
                    "education": education,
                    "age": int(age),
                    "parity": int(parity),
                    "induced": int(induced),
                    "spontaneous": int(spontaneous),
                }
            ]
        )[bundle["fitur_mentah"]]

        proba_kasus = float(bundle["pipeline"].predict_proba(input_df)[0, 1])
        proba_kontrol = 1.0 - proba_kasus
        prediksi = 1 if proba_kasus >= bundle["threshold"] else 0

        st.markdown("---")
        c1, c2, c3 = st.columns(3)
        with c1:
            if prediksi == 1:
                st.error("Hasil Klasifikasi: **Kasus Risiko Infertilitas (1)**")
            else:
                st.success("Hasil Klasifikasi: **Kontrol / Risiko Rendah (0)**")
        with c2:
            st.metric("Probabilitas Kasus (1)", f"{proba_kasus:.1%}")
        with c3:
            st.metric("Probabilitas Kontrol (0)", f"{proba_kontrol:.1%}")

with tab_evaluasi:
    st.subheader(f"Performa Model Terpilih: {bundle['nama_model']}")
    metrik = bundle["metrik_test"]

    m1, m2, m3, m4, m5 = st.columns(5)
    m1.metric("ROC-AUC (Test)", f"{metrik['roc_auc']:.3f}")
    m2.metric("Recall (Test)", f"{metrik['recall']:.3f}")
    m3.metric("F1-Score (Test)", f"{metrik['f1_score']:.3f}")
    m4.metric("Precision (Test)", f"{metrik['precision']:.3f}")
    m5.metric("Accuracy (Test)", f"{metrik['accuracy']:.3f}")

    st.markdown("**Perbandingan Model (5-Fold Stratified Cross-Validation di Data Train):**")
    st.dataframe(pd.DataFrame(bundle["perbandingan_cv"]), use_container_width=True)

    st.markdown("**Confusion Matrix pada Data Test (50 sampel):**")
    cm = metrik["confusion_matrix"]
    cm_df = pd.DataFrame(
        [[cm["TN"], cm["FP"]], [cm["FN"], cm["TP"]]],
        index=["Aktual: Kontrol (0)", "Aktual: Kasus (1)"],
        columns=["Prediksi: Kontrol (0)", "Prediksi: Kasus (1)"],
    )
    st.table(cm_df)

    st.caption(
        f"Threshold keputusan: {bundle['threshold']} | Versi scikit-learn: {bundle['versi_sklearn']}"
    )

Writing app.py


In [ ]:
import subprocess
import time
import re
import os

# 0. Pastikan streamlit terinstal di lingkungan Colab
!pip install -q streamlit

# 1. Pastikan file app.py ditulis dengan nama-nama komponen bahasa Indonesia yang mudah dimengerti
app_code = """
\"\"\"Aplikasi Web Streamlit untuk Prediksi Risiko Infertilitas (Kasus 8).\"\"\"
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st
import fitur  # Wajib di-import agar FunctionTransformer di pipeline terbaca saat load

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "models" / "model_infertilitas_kasus08.joblib"

@st.cache_resource
def load_bundle():
    if not MODEL_PATH.exists():
        return None
    return joblib.load(MODEL_PATH)

st.set_page_config(page_title="Prediksi Risiko Infertilitas", layout="wide")
st.title("Sistem Prediksi Risiko Infertilitas (Sukar Hamil)")
st.caption("Aplikasi Deteksi Dini Berbasis Kecerdasan Buatan · UTS Kasus 08 · NIM 23552011090")

bundle = load_bundle()
if bundle is None:
    st.error("Berkas model tidak ditemukan! Silakan jalankan latih.py terlebih dahulu.")
    st.stop()

tab_prediksi, tab_evaluasi = st.tabs(["╡╠ Form Prediksi Pasien Baru", "═║ Evaluasi & Performa Model"])

# Pemetaan pendidikan agar lebih ramah dibaca
map_pendidikan = {
    "0-5yrs": "SD / Tidak Sekolah (0-5 tahun masa sekolah)",
    "6-11yrs": "SMP / SMA (6-11 tahun masa sekolah)",
    "12+ yrs": "Diploma / Sarjana / Lebih Tinggi (12+ tahun masa sekolah)"
}

with tab_prediksi:
    st.subheader("Isi Riwayat Kesehatan & Kehamilan Pasien")
    st.markdown("Silakan isi formulir di bawah ini dengan data pasien yang ingin dianalisis:")

    col1, col2 = st.columns(2)
    with col1:
        edu_selected = st.selectbox(
            "1. Tingkat Pendidikan Terakhir",
            options=bundle["opsi_education"],
            format_func=lambda x: map_pendidikan.get(x, x)
        )
        age = st.number_input("2. Usia Pasien Saat Ini (Tahun)", min_value=15, max_value=60, value=30, step=1)
        parity = st.number_input("3. Jumlah Kelahiran Hidup Sebelumnya (Jumlah Anak)", min_value=0, max_value=15, value=1, step=1)

    with col2:
        induced = st.number_input("4. Jumlah Riwayat Keguguran Sengaja (Medis/Kuret)", min_value=0, max_value=10, value=0, step=1)
        spontaneous = st.number_input("5. Jumlah Riwayat Keguguran Alami (Tanpa Sengaja)", min_value=0, max_value=10, value=0, step=1)

    # Menghitung ringkasan secara otomatis
    total_kehamilan = parity + induced + spontaneous
    total_kehilangan = induced + spontaneous

    st.info(
        f"**Ringkasan Riwayat:** Total Pernah Hamil = **{total_kehamilan} kali** | "
        f"Total Riwayat Kehilangan (Keguguran) = **{total_kehilangan} kali**"
    )

    if st.button("Mulai Analisis Risiko", type="primary"):
        input_df = pd.DataFrame([{
            "education": edu_selected,
            "age": int(age),
            "parity": int(parity),
            "induced": int(induced),
            "spontaneous": int(spontaneous)
        }])[bundle["fitur_mentah"]]

        # Prediksi menggunakan pipeline model terbaik
        proba_kasus = float(bundle["pipeline"].predict_proba(input_df)[0, 1])
        proba_kontrol = 1.0 - proba_kasus
        hasil_prediksi = 1 if proba_kasus >= bundle["threshold"] else 0

        st.markdown("---")
        st.subheader("Hasil Analisis AI:")

        c1, c2 = st.columns(2)
        with c1:
            if hasil_prediksi == 1:
                st.error("★═ **HASIL: TERDETEKSI RISIKO TINGGI INFERTILITAS (KASUS)**")
                st.write("Pasien disarankan untuk melakukan konsultasi lebih lanjut dengan dokter spesialis kandungan (Sp.OG).")
            else:
                st.success("╔═ **HASIL: RISIKO RENDAH / NORMAL (KONTROL)**")
                st.write("Pasien saat ini tergolong dalam kategori risiko rendah terkait infertilitas berdasarkan riwayat kehamilannya.")

        with c2:
            st.metric("Tingkat Keyakinan Risiko Tinggi", f"{proba_kasus:.1%}")
            st.metric("Tingkat Keyakinan Normal", f"{proba_kontrol:.1%}")

with tab_evaluasi:
    st.subheader(f"Informasi Performa AI ({bundle['nama_model']})")
    st.write("Model ini dievaluasi menggunakan data uji independen yang tidak pernah dilihat sebelumnya untuk menjamin akurasi sesungguhnya.")

    metrik = bundle["metrik_test"]
    m1, m2, m3 = st.columns(3)
    m1.metric("Skor Akurasi Prediksi", f"{metrik['accuracy']:.1%}", help="Seberapa sering model memprediksi dengan benar keseluruhan data.")
    m2.metric("Sensitivitas (Recall)", f"{metrik['recall']:.1%}", help="Kemampuan model menemukan semua pasien yang benar-benar berisiko.")
    m3.metric("Skor ROC-AUC", f"{metrik['roc_auc']:.3f}", help="Kemampuan model membedakan pasien sehat vs berisiko tinggi.")

    st.markdown("**Tabel Perbandingan Akurasi Model (Hasil Evaluasi Lintas-Validasi):**")
    df_cv = pd.DataFrame(bundle["perbandingan_cv"])
    df_cv.columns = ["Nama Model", "Akurasi", "Presisi", "Recall", "F1-Score", "ROC-AUC"]
    st.dataframe(df_cv, use_container_width=True)
"""

with open("app.py", "w") as f:
    f.write(app_code)

# 2. Unduh Cloudflared jika belum siap
if not os.path.exists("cloudflared-linux-amd64"):
    !wget -q -O cloudflared-linux-amd64 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared-linux-amd64

# 3. Jalankan Streamlit kembali
with open("streamlit.log", "w") as log_file:
    proc_streamlit = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"], stdout=log_file, stderr=log_file)

time.sleep(5)

# 4. Jalankan Cloudflared Tunnel
with open("cloudflared.log", "w") as cf_log:
    proc_tunnel = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"], stdout=cf_log, stderr=cf_log)

time.sleep(7)

# 5. Cari dan tampilkan link trycloudflare
with open("cloudflared.log", "r") as cf_log:
    log_content = cf_log.read()
    links = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
    if links:
        print("\n=== APLIKASI WEB STREAMLIT ANDA SIAP! ===")
        print(f"Silakan klik link berikut untuk membuka aplikasi:\n{links[0]}")
        print("=========================================")
    else:
        print("Link belum siap atau gagal dimuat, berikut isi log cloudflared:")
        print(log_content)


=== APLIKASI WEB STREAMLIT ANDA SIAP! ===
Silakan klik link berikut untuk membuka aplikasi:
https://manitoba-reply-karl-alike.trycloudflare.com


In [ ]:
import pandas as pd

# Menggunakan variabel 'data' yang telah didefinisikan sebelumnya
data.to_csv('//content/drive/My Drive/UTS_Kasus08/hasil_dataset.csv', index=False)

In [16]:
%cd /content/drive/MyDrive/UTS_Kasus08

!echo "*.log" > .gitignore
!git init
!git branch -M main
!git config --global user.email "ipanmaulana123465@gmail.com"
!git config --global user.name "IpanTEKNIKINFORMATIKA"

!git remote add origin https://TOKEN_ANDA@github.com/UsernameGitHubAnda/nama-repo.git
!git add .
!git update-index --chmod=+x cloudflared-linux-amd64
!git commit -m "Upload proyek UTS Kasus 08 beserta dataset dan model"
!git push -u origin main

/content/drive/MyDrive/UTS_Kasus08
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/UTS_Kasus08/.git/
[main (root-commit) 9439c99] Upload proyek UTS Kasus 08 beserta dataset dan model
 8 files changed, 598 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 Dataset/kasus08_infert.csv
 create mode 100644 Models/model_infertilitas_kasus08.joblib
 create mode 100644 app.py
 create mode 100755 cloudflared-linux-amd64
 create mode 100644 fitur.py
 create mode 100644 kasus08_colab.ipynb
 creat

In [20]:
%cd /content/drive/MyDrive/UTS_Kasus08

!git remote set-url origin https://@github.com/IpanTEKNIKINFORMATIKA/UTS_Kasus08.git
!git push -u origin main

/content/drive/MyDrive/UTS_Kasus08
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (10/10), done.
Writing objects: 100% (12/12), 19.10 MiB | 1.81 MiB/s, done.
Total 12 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/IpanTEKNIKINFORMATIKA/UTS_Kasus08.git
 * [new branch]      main -> main
branch 'main' set up to track 'origin/main'.


In [28]:
%%bash
cd /content/drive/MyDrive/UTS_Kasus08

# Catatan: Pastikan Anda telah mengklik link unblock dari GitHub terlebih dahulu
# atau menghapus token mentah dari riwayat notebook sebelum melakukan push.

echo "=== Status Git Saat Ini ==="
git status

# Push ulang setelah masalah token diselesaikan / di-bypass via link GitHub
echo -e "\n=== Mencoba Push Kembali ke GitHub ==="
git push origin main

=== Status Git Saat Ini ===
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   kasus08_colab.ipynb

no changes added to commit (use "git add" and/or "git commit -a")

=== Mencoba Push Kembali ke GitHub ===


remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ——————————————————————        
remote:        locations:        
remote:          - commit: c7a084149d3e58963878e1dd949ab10447873ae4        
remote:            path: kasus08_colab.ipynb:1        
remote:             
remote:        (?) To push, remove secret from commit(s) or follow this URL t

CalledProcessError: Command 'b'cd /content/drive/MyDrive/UTS_Kasus08\n\n# Catatan: Pastikan Anda telah mengklik link unblock dari GitHub terlebih dahulu\n# atau menghapus token mentah dari riwayat notebook sebelum melakukan push.\n\necho "=== Status Git Saat Ini ==="\ngit status\n\n# Push ulang setelah masalah token diselesaikan / di-bypass via link GitHub\necho -e "\\n=== Mencoba Push Kembali ke GitHub ==="\ngit push origin main\n'' returned non-zero exit status 1.

In [29]:
%%bash
cd /content/drive/MyDrive/UTS_Kasus08

# 1. Batalkan commit terakhir yang membawa perubahan notebook berisi token rahasia
echo "=== Membatalkan Commit yang Melanggar Keamanan ==="
git reset --soft HEAD~1

# 2. Hapus token atau informasi sensitif dari konfigurasi lokal jika ada
git remote set-url origin https://github.com/IpanTEKNIKINFORMATIKA/UTS_Kasus08.git

# 3. Masukkan Token secara aman langsung ke URL remote hanya untuk sesi push ini
# (tidak akan tersimpan di dalam riwayat file notebook)
export GITHUB_TOKEN=""
git remote set-url origin https://${GITHUB_TOKEN}@github.com/IpanTEKNIKINFORMATIKA/UTS_Kasus08.git

# 4. Lakukan commit ulang dengan aman
echo "\n=== Melakukan Commit Ulang Secara Bersih ==="
git commit -m "Sinkronisasi aman proyek UTS Kasus 08"

# 5. Push kembali ke GitHub
echo "\n=== Mengirimkan Perubahan Bersih ke GitHub ==="
git push -f origin main


=== Membatalkan Commit yang Melanggar Keamanan ===
\n=== Melakukan Commit Ulang Secara Bersih ===
[main 5a9dd9a] Sinkronisasi aman proyek UTS Kasus 08
 2 files changed, 1 insertion(+), 1 deletion(-)
 mode change 100755 => 100644 cloudflared-linux-amd64
\n=== Mengirimkan Perubahan Bersih ke GitHub ===


remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ——————————————————————        
remote:        locations:        
remote:          - commit: 5a9dd9aea8c37390c81972156798f1f93cbfdc76        
remote:            path: kasus08_colab.ipynb:1        
remote:             
remote:        (?) To push, remove secret from commit(s) or follow this URL t

CalledProcessError: Command 'b'cd /content/drive/MyDrive/UTS_Kasus08\n\n# 1. Batalkan commit terakhir yang membawa perubahan notebook berisi token rahasia\necho "=== Membatalkan Commit yang Melanggar Keamanan ==="\ngit reset --soft HEAD~1\n\n# 2. Hapus token atau informasi sensitif dari konfigurasi lokal jika ada\ngit remote set-url origin https://github.com/IpanTEKNIKINFORMATIKA/UTS_Kasus08.git\n\n# 3. Masukkan Token secara aman langsung ke URL remote hanya untuk sesi push ini\n# (tidak akan tersimpan di dalam riwayat file notebook)\nexport GITHUB_TOKEN=""\ngit remote set-url origin https://${GITHUB_TOKEN}@github.com/IpanTEKNIKINFORMATIKA/UTS_Kasus08.git\n\n# 4. Lakukan commit ulang dengan aman\necho "\\n=== Melakukan Commit Ulang Secara Bersih ==="\ngit commit -m "Sinkronisasi aman proyek UTS Kasus 08"\n\n# 5. Push kembali ke GitHub\necho "\\n=== Mengirimkan Perubahan Bersih ke GitHub ==="\ngit push -f origin main\n'' returned non-zero exit status 1.